In [ ]:
import os
import io
import pandas as pd
from pathlib import Path
from openai import OpenAI

base_dir = Path(__file__).resolve().parents[2]
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY not set")
client = OpenAI(api_key=api_key)

icd9_top100_path = base_dir / "data" / "priors" / "top100_icd9.csv"
icd9_proc_master_path = base_dir / "data" / "input" / "D_ICD_PROCEDURES.csv"
output_path = base_dir / "outputs" / "synthetic" / "synthetic_ehr_full.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

icd9_top100_string = pd.read_csv(icd9_top100_path).to_string(index=False)
icd9_proc_master_string = pd.read_csv(icd9_proc_master_path).to_string(index=False)

def require_csv(text: str, expected_cols: list[str]) -> pd.DataFrame:
    df = pd.read_csv(io.StringIO(text))
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns {missing} in model output")
    ordered = expected_cols + [c for c in df.columns if c not in expected_cols]
    return df[ordered]

# Prompt 1: demographics
_demo_cols = [
    "AGE",
    "LANGUAGE",
    "RELIGION",
    "MARITAL_STATUS",
    "ETHNICITY",
    "INSURANCE",
    "HOSPITAL_EXPIRE_FLAG",
]
demo_prompt = f"""
You are a hospital intake expert generating realistic patient demographics for a synthetic EHR.
Use only reasoning and publicly available clinical knowledge. Do NOT use or infer real patient data.
Match these priors: { "mortality_rate": 0.14778, "demographics": { "LANGUAGE": { "ENGL": 0.56, "None": 0.36, "OTHER": 0.08 }, "RELIGION": { "CATHOLIC": 0.35, "NOT SPECIFIED": 0.21, "UNOBTAINABLE": 0.13, "PROTESTANT QUAKER": 0.12, "JEWISH": 0.09, "OTHER": 0.10 }, "MARITAL_STATUS": { "MARRIED": 0.48, "SINGLE": 0.24, "WIDOWED": 0.14, "None": 0.06, "DIVORCED": 0.06, "OTHER": 0.02 }, "ETHNICITY": { "WHITE": 0.70, "UNKNOWN/NOT SPECIFIED": 0.10, "BLACK/AFRICAN AMERICAN": 0.07, "HISPANIC OR LATINO": 0.02, "OTHER": 0.11 }, "INSURANCE": { "Medicare": 0.5254320098745114, "Private": 0.34712507714462043, "Medicaid": 0.0829818967290681, "Government": 0.030292120962764863, "Self Pay": 0.014168895289035179 }, "GENDER": { "M": 0.5656757868751285, "F": 0.4343242131248714 }, "AGE_BIN": { "65-79": 0.30636700267434686, "50-64": 0.270854762394569, "80+": 0.15104916683809916, "35-49": 0.1378060069944456, "18-34": 0.07398169101008023, "1-17": 0.059941370088459164 } } }.
Rules: patients aged 1-17 cannot be married, widowed, or divorced.
Output format: return ONLY CSV with header and exactly 100 rows. Columns: {', '.join(_demo_cols)}.
"""
resp = client.responses.create(
    model="gpt-5.1",
    input=demo_prompt,
    reasoning={"effort": "medium"},
).output_text

df_demo = require_csv(resp, _demo_cols)

# Prompt 2: main diagnosis
_diag_cols = _demo_cols + ["MAIN_DIAGNOSIS"]
diag_prompt = f"""
You are a professional medical coder assigning a primary diagnosis (ICD-9) to each synthetic patient based on demographics.
Use only public clinical knowledge (e.g., Mayo Clinic). Do not use real patient data.
Use ICD9 codes from this top-100 list:
{icd9_top100_string}

Patient demographics CSV:
{df_demo.to_csv(index=False)}

Output format: return ONLY CSV with the same rows, columns {', '.join(_diag_cols)} (add MAIN_DIAGNOSIS), no commentary.
"""
resp = client.responses.create(
    model="gpt-5.1",
    input=diag_prompt,
    reasoning={"effort": "medium"},
).output_text

df_diag = require_csv(resp, _diag_cols)

# Prompt 3: comorbidities
_comorb_cols = _diag_cols + ["COMORBIDITIES"]
comorb_prompt = f"""
Assign 0-6 ICD-9 comorbidities per patient based on demographics and main diagnosis.
Use only public clinical knowledge (e.g., Mayo Clinic). Do not use real patient data.
ICD9 top 100 list:
{icd9_top100_string}

Patient info CSV (includes MAIN_DIAGNOSIS):
{df_diag.to_csv(index=False)}

Output format: return ONLY CSV with the same rows, columns {', '.join(_comorb_cols)} (COMORBIDITIES as pipe- or semicolon-separated list), no commentary.
"""
resp = client.responses.create(
    model="gpt-5.1",
    input=comorb_prompt,
    reasoning={"effort": "medium"},
).output_text

df_comorb = require_csv(resp, _comorb_cols)

# Prompt 4: procedures
_proc_cols = _comorb_cols + ["PROCEDURES"]
proc_prompt = f"""
Assign up to 6 ICD-9 procedure codes per patient based on demographics, main diagnosis, and comorbidities.
Use only public clinical knowledge (e.g., Mayo Clinic). Use codes from this procedures master:
{icd9_proc_master_string}

Patient info CSV (includes MAIN_DIAGNOSIS and COMORBIDITIES):
{df_comorb.to_csv(index=False)}

Output format: return ONLY CSV with the same rows, columns {', '.join(_proc_cols)} (PROCEDURES as pipe- or semicolon-separated list), no commentary.
"""
resp = client.responses.create(
    model="gpt-5.1",
    input=proc_prompt,
    reasoning={"effort": "medium"},
).output_text

df_proc = require_csv(resp, _proc_cols)
df_proc.to_csv(output_path, index=False)
print(f"Saved synthetic EHR to {output_path}")
